# ENSO Forecasting: 1D CNN Regression

1D CNN model for next-month Nino 3.4 SST prediction from the 12-month lag window.

Validation is taken from the end of the training period to keep temporal order intact.


## 1. Import Libraries

If the import cell says TensorFlow is missing, run the install cell below once, then restart the notebook kernel before continuing.

In [ ]:
# Optional: run once if TensorFlow is missing.
# Restart the kernel after installation.
# %pip install tensorflow

In [ ]:
# AI-assisted: Initial structure and implementation were drafted with Codex.
# Human edits: Adapted for ENSO dataset, selected features/targets, verified outputs, and tuned evaluation.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import tensorflow as tf
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Conv1D, Dense, Flatten, Input, MaxPooling1D
    from tensorflow.keras.models import Sequential
except ModuleNotFoundError as error:
    if error.name == "tensorflow":
        raise ModuleNotFoundError(
            "TensorFlow is not installed in this notebook environment. "
            "Run `%pip install tensorflow`, restart the kernel, then run again."
        ) from error
    raise

np.random.seed(42)
tf.random.set_seed(42)

## 2. Load the Data

In [ ]:
# Locate the data folder from either the project root or notebooks folder.
data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

X_train_full = pd.read_csv(data_dir / "X_train.csv").to_numpy(dtype=np.float32)
X_test_raw = pd.read_csv(data_dir / "X_test.csv").to_numpy(dtype=np.float32)
y_train_full = pd.read_csv(data_dir / "y_train.csv").to_numpy(dtype=np.float32).ravel()
y_test = pd.read_csv(data_dir / "y_test.csv").to_numpy(dtype=np.float32).ravel()

print("X_train_full shape:", X_train_full.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_train_full shape:", y_train_full.shape)
print("y_test shape:", y_test.shape)

## 3. Prepare Data for Model

In [ ]:
val_fraction = 0.20
val_size = int(len(X_train_full) * val_fraction)
train_end = len(X_train_full) - val_size

X_train_raw = X_train_full[:train_end]
X_val_raw = X_train_full[train_end:]
y_train = y_train_full[:train_end]
y_val = y_train_full[train_end:]

print("CNN train shape:", X_train_raw.shape, y_train.shape)
print("CNN validation shape:", X_val_raw.shape, y_val.shape)
print("Final test shape:", X_test_raw.shape, y_test.shape)

In [ ]:
n_timesteps = X_train_raw.shape[1]
n_features = 1

X_train_cnn = X_train_raw.reshape((X_train_raw.shape[0], n_timesteps, n_features))
X_val_cnn = X_val_raw.reshape((X_val_raw.shape[0], n_timesteps, n_features))
X_test_cnn = X_test_raw.reshape((X_test_raw.shape[0], n_timesteps, n_features))

print("X_train_cnn shape:", X_train_cnn.shape)
print("X_val_cnn shape:", X_val_cnn.shape)
print("X_test_cnn shape:", X_test_cnn.shape)

## 4. Build the Model

In [ ]:
model = Sequential([
    Input(shape=(n_timesteps, n_features)),
    Conv1D(filters=64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation="relu"),
    Dense(1, activation="linear")
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()

## 5. Train the Model

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_cnn,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_cnn, y_val),
    callbacks=[early_stopping],
    verbose=1
)

## 6. Evaluate on the Test Set

In [ ]:
y_pred_train = model.predict(X_train_cnn).ravel()
y_pred_val = model.predict(X_val_cnn).ravel()
y_pred_test = model.predict(X_test_cnn).ravel()

def evaluate_regression(y_true, y_pred, split_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"--- {split_name} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R^2:  {r2:.4f}")

    return rmse, mae, r2

print("1D CNN Performance")
train_metrics = evaluate_regression(y_train, y_pred_train, "Train")
val_metrics = evaluate_regression(y_val, y_pred_val, "Validation")
test_metrics = evaluate_regression(y_test, y_pred_test, "Test")

## 7. Visualize Results

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("1D CNN Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(data_dir / "cnn_1d_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test, label="Actual", linewidth=2)
plt.plot(y_pred_test, label="1D CNN Predicted", linestyle="--", linewidth=2)
plt.xlabel("Test Sample")
plt.ylabel("Scaled SST")
plt.title("1D CNN Predictions vs Actual Test Values")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(data_dir / "cnn_1d_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Test Predictions and Metrics

In [ ]:
predictions_df = pd.DataFrame({
    "y_actual": y_test,
    "y_pred_CNN_1D": y_pred_test
})
predictions_df.to_csv(data_dir / "cnn_1d_test_predictions.csv", index=False)

metrics_df = pd.DataFrame({
    "Model": ["1D CNN"],
    "RMSE_Train": [train_metrics[0]],
    "MAE_Train": [train_metrics[1]],
    "R2_Train": [train_metrics[2]],
    "RMSE_Val": [val_metrics[0]],
    "MAE_Val": [val_metrics[1]],
    "R2_Val": [val_metrics[2]],
    "RMSE_Test": [test_metrics[0]],
    "MAE_Test": [test_metrics[1]],
    "R2_Test": [test_metrics[2]]
})
metrics_df.to_csv(data_dir / "cnn_1d_metrics.csv", index=False)

display(metrics_df)
print("Saved predictions to:", (data_dir / "cnn_1d_test_predictions.csv").resolve())
print("Saved metrics to:", (data_dir / "cnn_1d_metrics.csv").resolve())